# 01 — House index audit

## Objectif

Télécharger et parser les index XML House 2013–2026.

Produire :
- `house_filings_index.csv` ;
- `house_ptr_index.csv` ;
- `ptr_index_YYYY.csv` ;
- `reports/house_index_audit.md`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.house_index import (
    build_house_filings_index, filter_ptr_filings, save_index_outputs,
    summarize_house_index, write_house_index_report, build_house_zip_url,
    build_house_pdf_url,
)
from src.utils import PROCESSED_HOUSE_DIR, AUDIT_DIR

print("Imports OK")

In [ ]:
START_YEAR = 2013
END_YEAR = 2026
SLEEP_SECONDS = 0.25
FORCE_DOWNLOAD = False

print(build_house_zip_url(START_YEAR))
print(build_house_pdf_url(2026, "20034201"))

## Exécution

Les anciens chiffres sont des repères d’audit. Ils ne sont pas hardcodés.

In [ ]:
df_all, logs = build_house_filings_index(
    start_year=START_YEAR,
    end_year=END_YEAR,
    sleep_seconds=SLEEP_SECONDS,
    force=FORCE_DOWNLOAD,
)

print("df_all shape:", df_all.shape)
logs

In [ ]:
df_ptr = filter_ptr_filings(df_all)
print("df_ptr shape:", df_ptr.shape)

if not df_all.empty:
    display(df_all.head())
if not df_ptr.empty:
    display(df_ptr.head())

In [ ]:
summary = summarize_house_index(df_all, df_ptr, logs)
summary

In [ ]:
if not df_all.empty:
    display(pd.crosstab(df_all["year"], df_all["filing_type"]))

if not df_ptr.empty:
    display(df_ptr.groupby("year").size().rename("n_ptr"))

In [ ]:
checks = {
    "missing_doc_id": int((df_all["doc_id"].astype(str).str.strip() == "").sum()) if not df_all.empty else 0,
    "missing_filing_date": int(df_all["filing_date"].isna().sum()) if not df_all.empty else 0,
    "duplicates_year_doc_id": int(df_all.duplicated(["year", "doc_id"]).sum()) if not df_all.empty else 0,
}
checks

In [ ]:
output_paths = save_index_outputs(df_all, df_ptr, PROCESSED_HOUSE_DIR)
for name, path in output_paths.items():
    print(name, "->", Path(path).relative_to(ROOT))

In [ ]:
logs_path = AUDIT_DIR / "house_index_download_logs.csv"
logs.to_csv(logs_path, index=False)
report_path = write_house_index_report(summary, output_paths)

print("Written:", logs_path.relative_to(ROOT))
print("Written:", report_path.relative_to(ROOT))

## Conclusion

**OK** si `house_ptr_index.csv` existe et que les anomalies sont explicites.

**NEXT** — télécharger les PDF avec `02_house_pdf_download_manifest.ipynb`.